# 🔑 Notebook 2: Idempotency Keys

The client generates a unique key (a UUID) for each *logical* request and sends it in a header like `Idempotency-Key: <uuid>`. The server stores `key -> result` the first time. Subsequent retries with the same key get the **cached** result back — no side effect.

This is how Stripe, AWS, GitHub, Shopify, PayPal, and most modern APIs handle retries safely.

## 🧭 Mental model

```
  client                                server
  ------                                ------
  POST /charge                          
  Idempotency-Key: 9f3a…                
  { amount: 10 }          ───────►      first time? apply, cache, reply
                          ◄───────      200 { balance: 90 }   (response lost 😵)

  POST /charge (retry)                  
  Idempotency-Key: 9f3a…                
  { amount: 10 }          ───────►      seen key → return cached reply
                          ◄───────      200 { balance: 90 }   ← exactly-once effect
```


## 🛠️ Setup

```bash
cd 04-patterns/idempotency
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

## 🟩 GOOD: in-memory replay cache

In [ ]:
import uuid

class PaymentService:
    def __init__(self):
        self.balances = {'alice': 100}
        self.idem = {}  # key -> stored result

    def charge(self, key, account, amount):
        if key in self.idem:
            print(f'  ↩ replay for {key[:8]}… — returning cached response')
            return self.idem[key]
        self.balances[account] -= amount
        result = {'ok': True, 'balance': self.balances[account], 'charged': amount}
        self.idem[key] = result
        return result

svc = PaymentService()
k = str(uuid.uuid4())
print('1st:', svc.charge(k, 'alice', 10))
print('2nd:', svc.charge(k, 'alice', 10))
print('3rd:', svc.charge(k, 'alice', 10))
print(f"\nAlice ended with {svc.balances['alice']} — charged exactly once ✅")


A *different* key charges again — because that's a *different logical request*:

In [ ]:
k2 = str(uuid.uuid4())  # brand-new key, brand-new intent
print('new key:', svc.charge(k2, 'alice', 5))
print(f"Alice now at {svc.balances['alice']} — second *logical* charge applied.")


## 🟥 Gotcha 1: same key, *different* body

What if a buggy client reuses the same key but with a different amount? A naive cache would happily return the first result — silently ignoring the new request, and the client would think its $999 charge succeeded when nothing happened.

Stripe rejects this with a **`400`** ("Keys for idempotent requests can only be used with the same parameters"). We should too — loudly, because it always means a client bug.

The fix: store a **fingerprint** (hash) of the request body alongside the key, and compare on replay.

In [ ]:
import hashlib, json

class SaferPaymentService:
    def __init__(self):
        self.balances = {'alice': 100}
        self.idem = {}  # key -> (body_hash, result)

    @staticmethod
    def _fingerprint(body: dict) -> str:
        # sort_keys so {'a':1,'b':2} and {'b':2,'a':1} hash the same
        return hashlib.sha256(json.dumps(body, sort_keys=True).encode()).hexdigest()

    def charge(self, key, body):
        fp = self._fingerprint(body)
        if key in self.idem:
            cached_fp, cached_result = self.idem[key]
            if cached_fp != fp:
                raise ValueError(
                    '400 Bad Request: idempotency key reused with a different body'
                )
            return cached_result
        self.balances[body['account']] -= body['amount']
        result = {'ok': True, 'balance': self.balances[body['account']]}
        self.idem[key] = (fp, result)
        return result

svc = SaferPaymentService()
k = str(uuid.uuid4())
print('first  $10:', svc.charge(k, {'account': 'alice', 'amount': 10}))
print('replay $10:', svc.charge(k, {'account': 'alice', 'amount': 10}))
try:
    svc.charge(k, {'account': 'alice', 'amount': 999})  # SAME key, DIFFERENT body
except ValueError as e:
    print('rejected:', e)


## 🟥 Gotcha 2: keys must be scoped per caller

Here's the bug that turns an idempotency cache into a data leak. The key is a **client-supplied string**. If the server stores it in one global namespace, then any two callers who pick the same string share a cache entry — and the second one gets back the *first one's response*.

With UUIDs a collision is vanishingly unlikely by accident. It is not unlikely at all on purpose: nothing stops a caller from sending `Idempotency-Key: 1`.

In [ ]:
class GlobalKeyService:
    """BAD: one namespace for everyone's keys."""
    def __init__(self):
        self.balances = {'alice': 100, 'mallory': 100}
        self.idem = {}

    def charge(self, key, account, amount):
        if key in self.idem:
            return self.idem[key]                 # whose result is this?
        self.balances[account] -= amount
        result = {'account': account, 'balance': self.balances[account]}
        self.idem[key] = result
        return result

svc = GlobalKeyService()
print("alice   charges $10 with key '1':", svc.charge('1', 'alice', 10))
print("mallory charges $10 with key '1':", svc.charge('1', 'mallory', 10))
print()
print(f"mallory's balance is still {svc.balances['mallory']} — her charge silently vanished,")
print("and she was handed alice's balance in the response. One line of client-controlled")
print("input, and we have both a correctness bug and an information leak.")

### 🟩 Fix: the storage key is `(caller, idempotency_key)`

Scope every key to the authenticated principal. The client controls only *its half* of the namespace, so no client can reach into another's.

In [ ]:
class ScopedKeyService:
    """GOOD: the caller identity is part of the storage key."""
    def __init__(self):
        self.balances = {'alice': 100, 'mallory': 100}
        self.idem = {}

    def charge(self, caller, key, account, amount):
        storage_key = (caller, key)               # <- the whole fix
        if storage_key in self.idem:
            return {**self.idem[storage_key], 'replay': True}
        self.balances[account] -= amount
        result = {'account': account, 'balance': self.balances[account]}
        self.idem[storage_key] = result
        return {**result, 'replay': False}

svc = ScopedKeyService()
print("alice   key '1':", svc.charge('alice',   '1', 'alice',   10))
print("mallory key '1':", svc.charge('mallory', '1', 'mallory', 10))
print("alice   key '1' again (a real retry):", svc.charge('alice', '1', 'alice', 10))
print()
print("Same key string, different callers, no interference — and alice's own retry")
print("still replays correctly. Scope by whatever you authenticate: API key, user id,")
print("or tenant id. In a multi-tenant system, scope by tenant AND user.")

## 🧹 Operational notes

- **Put the key in a header** (`Idempotency-Key`) so middleware can enforce it uniformly — individual handlers shouldn't have to remember to check.
- **Scope it** to the authenticated caller (above). The client owns only its half of the namespace.
- **Expire it.** The replay cache grows forever otherwise. Size the TTL to the **retry window** you actually support — Stripe keeps keys 24h. Longer costs storage; shorter means a late retry is treated as a fresh request, which for a charge means a second charge. 24h is a bet that no sane client retries a day later.
- **Store the response, not just a "seen" flag.** A retry that gets `204 No Content` back instead of the original `{"charge_id": …}` has technically been deduplicated and is still useless — the client never learns the charge id.
- **Decide what to cache on failure.** If the first attempt returned a `500`, caching that response means the client can *never* retry successfully. The usual rule: cache `2xx` and `4xx` (deterministic outcomes), release the key on `5xx`.
- **In-memory only works for one process.** Multiple replicas + restarts ⇒ notebook 3 (durable, transactional dedup).